In [ ]:
!pip install datasets transformers torch scikit-learn -q
print("Libraries installed.")

In [ ]:
from huggingface_hub import login
login(token="***REMOVED***")
print("Logged in.")

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")

In [ ]:
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")

def get_variety_splits(variety):
    train = ds['train'].filter(lambda x: x['variety'] == variety)
    val   = ds['validation'].filter(lambda x: x['variety'] == variety)
    test  = ds['test'].filter(lambda x: x['variety'] == variety)
    return train, val, test

train_uk, val_uk, test_uk = get_variety_splits('en-UK')
train_in, val_in, test_in = get_variety_splits('en-IN')
train_au, val_au, test_au = get_variety_splits('en-AU')

# Combined dataset — all 3 varieties merged into one training set
train_combined = concatenate_datasets([train_uk, train_in, train_au])
val_combined   = concatenate_datasets([val_uk, val_in, val_au])

print(f"en-UK train: {len(train_uk)}")
print(f"en-IN train: {len(train_in)}")
print(f"en-AU train: {len(train_au)}")
print(f"Combined train: {len(train_combined)}")
print(f"Combined val: {len(val_combined)}")

In [ ]:
DEBERTA_MODEL = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

def prepare_dataset(dataset, label_column):
    def add_labels(x):
        return {'labels': int(x[label_column])}
    dataset = dataset.map(add_labels)
    dataset = dataset.map(tokenize, batched=True)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return dataset

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='macro')
    return {'macro_f1': f1}

print("DeBERTa tokenizer loaded.")
print(f"Model: {DEBERTA_MODEL}")

In [ ]:
from torch import nn

def train_deberta(train_data, val_data, label_column, variety_name, seed=1):
    print(f"\nTraining DeBERTa on {variety_name} for {label_column} (seed={seed})...")

    train_ds = prepare_dataset(train_data, label_column)
    val_ds   = prepare_dataset(val_data, label_column)

    model = AutoModelForSequenceClassification.from_pretrained(
        DEBERTA_MODEL,
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    args = TrainingArguments(
        output_dir=f'./results_deberta_{variety_name}_{label_column}_seed{seed}',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print(f"Done — DeBERTa on {variety_name} for {label_column} seed={seed}")
    return trainer

print("DeBERTa training function ready.")

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = torch.tensor([1.0, 3.0]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def train_deberta_sarcasm(train_data, val_data, variety_name, seed=1):
    print(f"\nTraining DeBERTa on {variety_name} for Sarcasm (seed={seed})...")

    train_ds = prepare_dataset(train_data, 'Sarcasm')
    val_ds   = prepare_dataset(val_data, 'Sarcasm')

    model = AutoModelForSequenceClassification.from_pretrained(
        DEBERTA_MODEL,
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    args = TrainingArguments(
        output_dir=f'./results_deberta_{variety_name}_sarcasm_seed{seed}',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        seed=seed,
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print(f"Done — DeBERTa on {variety_name} Sarcasm seed={seed}")
    return trainer

print("DeBERTa sarcasm training function ready.")

In [ ]:
def evaluate_model(trainer, test_data, label_column, train_variety, test_variety):
    test_ds = prepare_dataset(test_data, label_column)
    predictions = trainer.predict(test_ds)
    preds  = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    report = classification_report(
        labels, preds,
        target_names=['Negative', 'Positive'] if label_column == 'Sentiment'
                     else ['Not Sarcastic', 'Sarcastic'],
        output_dict=True
    )

    macro_f1 = report['macro avg']['f1-score']

    print(f"\nTrained on {train_variety} → Tested on {test_variety}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(classification_report(
        labels, preds,
        target_names=['Negative', 'Positive'] if label_column == 'Sentiment'
                     else ['Not Sarcastic', 'Sarcastic']
    ))

    return macro_f1, preds, labels

print("Evaluation function ready.")

In [ ]:
varieties_info = [
    ('en-UK', train_uk, val_uk),
    ('en-IN', train_in, val_in),
    ('en-AU', train_au, val_au),
    ('ALL',   train_combined, val_combined),
]

results_deberta_sent = {}

for variety_name, train_data, val_data in varieties_info:
    scores = []
    trainers = []
    for seed in [1, 2]:
        print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")
        trainer = train_deberta(train_data, val_data, 'Sentiment', variety_name, seed)
        trainers.append(trainer)

        # Evaluate on all 3 test sets
        seed_scores = {}
        for test_var, test_data in [('en-UK', test_uk), ('en-IN', test_in), ('en-AU', test_au)]:
            f1, _, _ = evaluate_model(trainer, test_data, 'Sentiment', variety_name, test_var)
            seed_scores[test_var] = f1
        scores.append(seed_scores)
        print(f"Seed {seed} scores: {seed_scores}")

    # Calculate mean and std for each test variety
    for test_var in ['en-UK', 'en-IN', 'en-AU']:
        s = [scores[0][test_var], scores[1][test_var]]
        results_deberta_sent[(variety_name, test_var)] = {
            'mean': np.mean(s),
            'std': np.std(s)
        }
        print(f"\n{variety_name} → {test_var}: {np.mean(s):.4f} ± {np.std(s):.4f}")

    # Save best trainer (seed 1)
    results_deberta_sent[f'trainer_{variety_name}'] = trainers[0]

print("\nAll DeBERTa Sentiment training done!")

In [ ]:
results_deberta_sarc = {}

for variety_name, train_data, val_data in varieties_info:
    scores = []
    trainers = []
    for seed in [1, 2]:
        print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")
        trainer = train_deberta_sarcasm(train_data, val_data, variety_name, seed)
        trainers.append(trainer)

        seed_scores = {}
        for test_var, test_data in [('en-UK', test_uk), ('en-IN', test_in), ('en-AU', test_au)]:
            f1, _, _ = evaluate_model(trainer, test_data, 'Sarcasm', variety_name, test_var)
            seed_scores[test_var] = f1
        scores.append(seed_scores)
        print(f"Seed {seed} scores: {seed_scores}")

    for test_var in ['en-UK', 'en-IN', 'en-AU']:
        s = [scores[0][test_var], scores[1][test_var]]
        results_deberta_sarc[(variety_name, test_var)] = {
            'mean': np.mean(s),
            'std': np.std(s)
        }
        print(f"\n{variety_name} → {test_var}: {np.mean(s):.4f} ± {np.std(s):.4f}")

    results_deberta_sarc[f'trainer_{variety_name}'] = trainers[0]

print("\nAll DeBERTa Sarcasm training done!")

In [ ]:
import shutil
import os

# Delete all checkpoint folders to free up disk space
# We only need the final results not the intermediate checkpoints
dirs_to_delete = [d for d in os.listdir('./') if d.startswith('results_deberta')]

for d in dirs_to_delete:
    shutil.rmtree(d)
    print(f"Deleted: {d}")

print("Disk space freed!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

varieties_labels = ['en-UK', 'en-IN', 'en-AU']
train_labels = ['en-UK', 'en-IN', 'en-AU', 'ALL']

sent_matrix = np.array([
    [results_deberta_sent[('en-UK','en-UK')]['mean'], results_deberta_sent[('en-UK','en-IN')]['mean'], results_deberta_sent[('en-UK','en-AU')]['mean']],
    [results_deberta_sent[('en-IN','en-UK')]['mean'], results_deberta_sent[('en-IN','en-IN')]['mean'], results_deberta_sent[('en-IN','en-AU')]['mean']],
    [results_deberta_sent[('en-AU','en-UK')]['mean'], results_deberta_sent[('en-AU','en-IN')]['mean'], results_deberta_sent[('en-AU','en-AU')]['mean']],
    [results_deberta_sent[('ALL','en-UK')]['mean'],   results_deberta_sent[('ALL','en-IN')]['mean'],   results_deberta_sent[('ALL','en-AU')]['mean']],
])

sarc_matrix = np.array([
    [results_deberta_sarc[('en-UK','en-UK')]['mean'], results_deberta_sarc[('en-UK','en-IN')]['mean'], results_deberta_sarc[('en-UK','en-AU')]['mean']],
    [results_deberta_sarc[('en-IN','en-UK')]['mean'], results_deberta_sarc[('en-IN','en-IN')]['mean'], results_deberta_sarc[('en-IN','en-AU')]['mean']],
    [results_deberta_sarc[('en-AU','en-UK')]['mean'], results_deberta_sarc[('en-AU','en-IN')]['mean'], results_deberta_sarc[('en-AU','en-AU')]['mean']],
    [results_deberta_sarc[('ALL','en-UK')]['mean'],   results_deberta_sarc[('ALL','en-IN')]['mean'],   results_deberta_sarc[('ALL','en-AU')]['mean']],
])

for ax, matrix, title in zip(axes,
    [sent_matrix, sarc_matrix],
    ['DeBERTa Sentiment\nMean Macro F1 (2 seeds)',
     'DeBERTa Sarcasm\nMean Macro F1 (2 seeds)']):

    sns.heatmap(matrix, annot=True, fmt='.4f', cmap='RdYlGn',
                xticklabels=[f'Test\n{v}' for v in varieties_labels],
                yticklabels=[f'Train\n{v}' for v in train_labels],
                vmin=0.4, vmax=1.0,
                linewidths=0.5, ax=ax,
                annot_kws={'size': 12, 'weight': 'bold'})
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Tested On', fontsize=10)
    ax.set_ylabel('Trained On', fontsize=10)

fig.suptitle('DeBERTa-v3-base Cross-Variety Evaluation Matrix\nIncluding Combined Training (ALL)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('deberta_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 65)
print("FINAL SUMMARY — DeBERTa Cross-Variety Evaluation")
print("=" * 65)

print("\nSENTIMENT (Mean ± Std across 2 seeds):")
print(f"{'Train':<8} {'Test':<8} {'Mean':>8} {'Std':>8}")
print("-" * 35)
for train_var in ['en-UK', 'en-IN', 'en-AU', 'ALL']:
    for test_var in ['en-UK', 'en-IN', 'en-AU']:
        r = results_deberta_sent[(train_var, test_var)]
        print(f"{train_var:<8} {test_var:<8} {r['mean']:>8.4f} {r['std']:>8.4f}")
    print()

print("\nSARCASM (Mean ± Std across 2 seeds):")
print(f"{'Train':<8} {'Test':<8} {'Mean':>8} {'Std':>8}")
print("-" * 35)
for train_var in ['en-UK', 'en-IN', 'en-AU', 'ALL']:
    for test_var in ['en-UK', 'en-IN', 'en-AU']:
        r = results_deberta_sarc[(train_var, test_var)]
        print(f"{train_var:<8} {test_var:<8} {r['mean']:>8.4f} {r['std']:>8.4f}")
    print()

print("=" * 65)